# Strategic Plan Funding Match Pipeline 

This notebook implements the complete matching workflow from strategic plans to stakeholder-ready funding recommendations.

**Phase 1** retrieves and ranks candidate calls using semantic evidence from a persistent vector database.

**Phase 2** adds explainability by decomposing scores, surfacing matched themes, and detecting strategic gaps.

**Phase 3** generates grounded executive summaries with citations and confidence labels.

The notebook’s role is to provide a reproducible, inspectable, end-to-end analysis pipeline.

## Imports

Import the libraries needed.


In [29]:
from pathlib import Path
from typing import Dict, List

import json
from datetime import datetime, timezone

import chromadb
import pandas as pd
from chromadb.utils import embedding_functions
from pypdf import PdfReader
from tqdm.auto import tqdm

from collections import Counter


from pipeline_core_methods import (
    build_call_xai as core_build_call_xai,
    chunk_text as core_chunk_text,
    clean_text as core_clean_text,
    rank_calls_for_sp_query as core_rank_calls_for_sp_query,
    similarity_from_distance as core_similarity_from_distance,
)


# Phase 1: Retrieval and Ranking

In this phase we retrieve relevant call evidence for each strategic plan and rank calls into a top-3 shortlist.

## Configuration

Defining global constants for paths, model/collection names, ranking thresholds, and expected strategic plan files.

In [11]:
PROJECT_ROOT = Path.cwd()
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma"
COLLECTION_NAME = "funding_calls"
EMBED_MODEL = "Qwen/Qwen3-Embedding-0.6B"

STRATEGIC_PLAN_DIR = PROJECT_ROOT / "docs" / "strategicplans"
EXPECTED_SP_FILES = {
    "Naples_Federico_Piano_strategico_2021_2026.pdf",
    "Politechnico_di_Milano_2023-2025.pdf",
    "Piano Strategico Luiss 2021-2024 (per sito).pdf",
    "Bocconi_Piano_Strategico2021-2025&Vision2030.pdf",
    "LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf",
    "Sapienza_2021_2027.pdf",
}

N_RESULTS = 30
TOP_K_CALLS = 3
EVIDENCE_PER_CALL = 3
STRONG_HIT_THRESHOLD = 0.55

SP_CHUNK_SIZE = 1200
SP_CHUNK_OVERLAP = 200
SP_MIN_CHUNK_CHARS = 160

print(f"Project root: {PROJECT_ROOT}")
print(f"Chroma dir: {CHROMA_DIR}")
print(f"Collection: {COLLECTION_NAME}")
print(f"SP directory: {STRATEGIC_PLAN_DIR}")
print(f"SP chunking: size={SP_CHUNK_SIZE}, overlap={SP_CHUNK_OVERLAP}, min_chars={SP_MIN_CHUNK_CHARS}")


Project root: /home/lburmazevic/Projects/EY-AI-Techniques-Solution
Chroma dir: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/data/chroma
Collection: funding_calls
SP directory: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/docs/strategicplans
SP chunking: size=1200, overlap=200, min_chars=160


## Strategic Plans

Load the six strategic plan PDFs, cleans raw text, and build overlapping chunks used as retrieval queries. It creates the project’s query-side representation, which directly influences retrieval relevance and call-level scoring


In [12]:
def clean_text(text: str) -> str:
    return core_clean_text(text)


def chunk_text(text: str, size: int = SP_CHUNK_SIZE, overlap: int = SP_CHUNK_OVERLAP) -> List[str]:
    return core_chunk_text(text, size=size, overlap=overlap, min_chars=SP_MIN_CHUNK_CHARS)


def extract_sp_text(pdf_path: Path) -> Dict:
    reader = PdfReader(str(pdf_path))
    page_texts: List[str] = []

    for page in reader.pages:
        text = clean_text(page.extract_text() or "")
        if len(text) >= 40:
            page_texts.append(text)

    sp_text = "\n".join(page_texts)
    sp_chunks = chunk_text(sp_text)

    return {
        "sp_id": pdf_path.stem,
        "source_file": pdf_path.name,
        "text": sp_text,
        "chunks": sp_chunks,
        "non_empty_pages": len(page_texts),
    }


sp_paths = sorted(STRATEGIC_PLAN_DIR.glob("*.pdf"))
found = {p.name for p in sp_paths}
missing = EXPECTED_SP_FILES - found
if missing:
    raise FileNotFoundError(f"Missing strategic plan files: {sorted(missing)}")

strategic_plans = []
for path in tqdm(sp_paths, desc="Loading strategic plans"):
    if path.name in EXPECTED_SP_FILES:
        strategic_plans.append(extract_sp_text(path))

print(f"Loaded {len(strategic_plans)} strategic plans")
for sp in strategic_plans:
    print(f" - {sp['source_file']}: non_empty_pages={sp['non_empty_pages']}, chars={len(sp['text'])}, chunks={len(sp['chunks'])}")



Loading strategic plans: 100%|██████████| 6/6 [00:08<00:00,  1.48s/it]

Loaded 6 strategic plans
 - Bocconi_Piano_Strategico2021-2025&Vision2030.pdf: non_empty_pages=38, chars=85284, chunks=86
 - LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf: non_empty_pages=74, chars=195991, chunks=196
 - Naples_Federico_Piano_strategico_2021_2026.pdf: non_empty_pages=68, chars=68387, chunks=69
 - Piano Strategico Luiss 2021-2024 (per sito).pdf: non_empty_pages=62, chars=55615, chunks=56
 - Politechnico_di_Milano_2023-2025.pdf: non_empty_pages=29, chars=29591, chunks=30
 - Sapienza_2021_2027.pdf: non_empty_pages=31, chars=58569, chunks=59


## Connect to Chroma

Attach to the persistent ChromaDB collection generated during ingestion.


In [13]:
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
collection = client.get_collection(name=COLLECTION_NAME, embedding_function=embed_fn)
print("Chroma collection is ready")


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 2155.18it/s]


Chroma collection is ready


## Ranking Logic

The objective is to rank funding calls by alignment with each strategic plan.

**Score design:**

- **semantic_score**: mean of top similarities (signal strength of best semantic matches).

- **coverage_score**: normalized breadth of call pages matched (signal diversity across the call).

- **consistency_score**: proportion of strong hits above threshold (signal stability across retrievals).

- **final_score**: weighted combination (0.7*semantic + 0.2*coverage + 0.1*consistency).

**Output**: top-3 ranked calls plus evidence chunks and score breakdown for each SP.

In [14]:
def similarity_from_distance(distance: float) -> float:
    return core_similarity_from_distance(distance)


def rank_calls_for_sp(sp: Dict, n_results: int = N_RESULTS) -> Dict:
    return core_rank_calls_for_sp_query(
        collection=collection,
        sp=sp,
        n_results=n_results,
        strong_hit_threshold=STRONG_HIT_THRESHOLD,
        evidence_per_call=EVIDENCE_PER_CALL,
        top_k_calls=TOP_K_CALLS,
    )



## Batch Run

Runs the ranking pipeline for all SPs and return a compact table of top recommendations.


In [15]:
phase1_results = []
for sp in tqdm(strategic_plans, desc='Ranking funding calls'):
    phase1_results.append(rank_calls_for_sp(sp))

rows = []
for result in phase1_results:
    for idx, call in enumerate(result['top_calls'], start=1):
        rows.append({
            'sp_file': result['sp_file'],
            'rank': idx,
            'call_name': call['call_name'],
            'final_score': call['final_score'],
            'semantic_score': call['semantic_score'],
            'coverage_score': call['coverage_score'],
            'consistency_score': call['consistency_score'],
        })

ranking_df = pd.DataFrame(rows).sort_values(['sp_file', 'rank']).reset_index(drop=True)
ranking_df


Ranking funding calls: 100%|██████████| 6/6 [46:58<00:00, 469.82s/it]


,sp_file,rank,call_name,final_score,semantic_score,coverage_score,consistency_score
0,Bocconi_Piano_Strategico2021-2025&Vision2030.pdf,1,PRIN 2022 PNRR.pdf,0.7728,0.6754,1.0,1.0
1,Bocconi_Piano_Strategico2021-2025&Vision2030.pdf,2,Ecosistemi dell'Innovazione PNRR.pdf,0.7726,0.6752,1.0,1.0
2,Bocconi_Piano_Strategico2021-2025&Vision2030.pdf,3,Partenariati Estesi PNRR.pdf,0.7712,0.6731,1.0,1.0
3,LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf,1,Ecosistemi dell'Innovazione PNRR.pdf,0.8047,0.7211,1.0,1.0
4,LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf,2,Partenariati Estesi PNRR.pdf,0.8036,0.7194,1.0,1.0
5,LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf,3,PRIN PNRR 2022 NextGen.pdf,0.7979,0.7113,1.0,1.0
6,Naples_Federico_Piano_strategico_2021_2026.pdf,1,PRIN 2022 PNRR.pdf,0.8248,0.7497,1.0,1.0
7,Naples_Federico_Piano_strategico_2021_2026.pdf,2,PRIN PNRR 2022 NextGen.pdf,0.8240,0.7486,1.0,1.0
8,Naples_Federico_Piano_strategico_2021_2026.pdf,3,Partenariati Estesi PNRR.pdf,0.8157,0.7367,1.0,1.0
9,Piano Strategico Luiss 2021-2024 (per sito).pdf,1,PRIN 2022 PNRR.pdf,0.8000,0.7143,1.0,1.0


## Evidence Preview

Inspect the top supporting chunks behind each recommended call for quick explainability checks.


In [16]:
def print_evidence_for_sp(sp_file: str):
    matches = [x for x in phase1_results if x['sp_file'] == sp_file]
    if not matches:
        print(f'No result for {sp_file}')
        return

    result = matches[0]
    print(f"Strategic plan: {result['sp_file']}")
    for i, call in enumerate(result['top_calls'], start=1):
        print(f"\nRank {i}: {call['call_name']} | final_score={call['final_score']}")
        for ev in call['evidence']:
            page_label = ev['page'] if ev['page'] is not None else 'n/a'
            print(f" - page={page_label}, sim={ev['similarity']}: {ev['chunk'][:180]}...")

# Example:
print_evidence_for_sp(phase1_results[0]['sp_file'])


Strategic plan: Bocconi_Piano_Strategico2021-2025&Vision2030.pdf

Rank 1: PRIN 2022 PNRR.pdf | final_score=0.7728
 - page=32, sim=0.6981: Ministero dell’Università e della Ricerca Direzione Generale della Ricerca Ufficio III 32 procedura di finanziamento, di avere beneficiato dei fondi dell’Unione Europea – Next Gene...
 - page=16, sim=0.6748: Ministero dell’Università e della Ricerca Direzione Generale della Ricerca Ufficio III 16 strategici emergenti correlati agli obiettivi di un cluster del Programma quadro europeo d...
 - page=16, sim=0.671: Ministero dell’Università e della Ricerca Direzione Generale della Ricerca Ufficio III 16 strategici emergenti correlati agli obiettivi di un cluster del Programma quadro europeo d...

Rank 2: Ecosistemi dell'Innovazione PNRR.pdf | final_score=0.7726
 - page=11, sim=0.6797: 11 privati, in maniera stabile e continuativa; b. trasferimento tecnologico e valorizzazione dei risultati della ricerca; c. supporto alla nascita e sviluppo di start -up e

# Phase 2: XAI Configuration

Explain why calls match and where SPs under-address call priorities.

In [17]:
XAI_TOP_THEMES = 4
XAI_EVIDENCE_PER_CALL = 3
XAI_MAX_GAPS = 4

GAP_MIN_CALL_SIGNAL = 0.2
GAP_MAX_SP_SIGNAL = 0.1


## Theme Lexicon

Map broad strategic themes to multilingual keywords for transparent, deterministic theme matching.


In [18]:
THEME_LEXICON = {
    "sustainability_climate": [
        "sustainability", "sustainable", "climate", "green", "decarbonization",
        "circular economy", "energy transition", "biodiversity", "sdg",
        "sostenibilita", "sostenibile", "clima", "transizione energetica",
        "economia circolare", "decarbonizzazione", "biodiversita"
    ],
    "ai_data_digital": [
        "artificial intelligence", "ai", "machine learning", "data", "digital",
        "digitalization", "digitization", "big data", "cloud", "cybersecurity",
        "intelligenza artificiale", "apprendimento automatico", "dati", "digitale",
        "digitalizzazione", "transizione digitale"
    ],
    "internationalization_collaboration": [
        "international", "internationalization", "cross-border", "european", "consortium",
        "partnership", "collaboration", "mobility", "network", "joint",
        "internazionale", "internazionalizzazione", "europeo", "consorzio",
        "partenariato", "collaborazione", "mobilita", "rete", "congiunto"
    ],
    "innovation_transfer_industry": [
        "innovation", "technology transfer", "valorization", "startup", "spin-off",
        "industry", "industrial", "commercialization", "patent",
        "innovazione", "trasferimento tecnologico", "valorizzazione",
        "industria", "industriale", "brevett"
    ],
    "skills_education_capacity": [
        "skills", "competence", "training", "education", "lifelong learning",
        "upskilling", "reskilling", "capacity building", "doctoral", "curriculum",
        "competenze", "formazione", "istruzione", "apprendimento permanente", "dottorato"
    ],
    "inclusion_gender_social": [
        "inclusion", "equity", "equality", "gender", "diversity", "accessibility",
        "social impact", "vulnerable", "cohesion",
        "inclusione", "equita", "uguaglianza", "genere", "diversita",
        "accessibilita", "impatto sociale", "vulnerabili", "coesione"
    ],
    "research_infrastructure_excellence": [
        "research infrastructure", "infrastructure", "laboratory", "equipment",
        "excellence", "scientific excellence", "facility", "platform",
        "infrastruttura di ricerca", "infrastruttura", "laboratorio", "attrezzature",
        "eccellenza", "eccellenza scientifica", "piattaforma"
    ],
    "governance_policy_reform": [
        "governance", "reform", "policy", "regulation", "institutional",
        "coordination", "roadmap", "implementation", "monitoring", "evaluation",
        "riforma", "politica", "regolazione", "istituzionale",
        "coordinamento", "attuazione", "monitoraggio", "valutazione"
    ],
}


## XAI Builders

Convert evidence chunks into matched themes and actionable gaps for each recommended funding call.


In [19]:
def build_call_xai(sp_text: str, call_row: Dict) -> Dict:
    return core_build_call_xai(
        sp_text=sp_text,
        call_row=call_row,
        theme_lexicon=THEME_LEXICON,
        xai_evidence_per_call=XAI_EVIDENCE_PER_CALL,
        xai_top_themes=XAI_TOP_THEMES,
        gap_min_call_signal=GAP_MIN_CALL_SIGNAL,
        gap_max_sp_signal=GAP_MAX_SP_SIGNAL,
        xai_max_gaps=XAI_MAX_GAPS,
    )



## Build

Attach explainability objects to each top-3 recommendation for every strategic plan.


In [20]:
phase2_results = []

for result in phase1_results:
    sp = next((item for item in strategic_plans if item["source_file"] == result["sp_file"]), None)
    if sp is None:
        continue

    calls_with_xai = []
    for call in result["top_calls"]:
        call_copy = dict(call)
        call_copy["xai"] = build_call_xai(sp["text"], call_copy)
        calls_with_xai.append(call_copy)

    phase2_results.append({
        "sp_id": result["sp_id"],
        "sp_file": result["sp_file"],
        "raw_hits": result["raw_hits"],
        "top_calls": calls_with_xai,
    })

print(f"Built Phase 2 XAI objects for {len(phase2_results)} strategic plans")


Built Phase 2 XAI objects for 6 strategic plans


## Summary Table

Show top-3 calls with matched themes and improvement actions in one compact output table.


In [21]:
xai_rows = []

for result in phase2_results:
    for rank, call in enumerate(result["top_calls"], start=1):
        xai = call["xai"]
        themes = ", ".join([item["theme"] for item in xai["matched_themes"]])
        actions = " | ".join([item["action"] for item in xai["key_gaps"]])

        xai_rows.append({
            "sp_file": result["sp_file"],
            "rank": rank,
            "call_name": xai["call_name"],
            "final_score": xai["final_score"],
            "matched_themes": themes,
            "improvement_actions": actions,
        })

xai_df = pd.DataFrame(xai_rows).sort_values(["sp_file", "rank"]).reset_index(drop=True)
xai_df


,sp_file,rank,call_name,final_score,matched_themes,improvement_actions
0,Bocconi_Piano_Strategico2021-2025&Vision2030.pdf,1,PRIN 2022 PNRR.pdf,0.7728,"internationalization_collaboration, innovation...",SP should strengthen measurable sustainability...
1,Bocconi_Piano_Strategico2021-2025&Vision2030.pdf,2,Ecosistemi dell'Innovazione PNRR.pdf,0.7726,"skills_education_capacity, innovation_transfer...",SP should strengthen technology transfer and i...
2,Bocconi_Piano_Strategico2021-2025&Vision2030.pdf,3,Partenariati Estesi PNRR.pdf,0.7712,"ai_data_digital, inclusion_gender_social, sust...",SP should strengthen inclusion and gender/soci...
3,LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf,1,Ecosistemi dell'Innovazione PNRR.pdf,0.8047,"ai_data_digital, internationalization_collabor...",
4,LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf,2,Partenariati Estesi PNRR.pdf,0.8036,"innovation_transfer_industry, governance_polic...",SP should strengthen governance and implementa...
5,LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf,3,PRIN PNRR 2022 NextGen.pdf,0.7979,"ai_data_digital, skills_education_capacity, su...",
6,Naples_Federico_Piano_strategico_2021_2026.pdf,1,PRIN 2022 PNRR.pdf,0.8248,"skills_education_capacity, governance_policy_r...",SP should strengthen governance and implementa...
7,Naples_Federico_Piano_strategico_2021_2026.pdf,2,PRIN PNRR 2022 NextGen.pdf,0.8240,"ai_data_digital, innovation_transfer_industry,...",
8,Naples_Federico_Piano_strategico_2021_2026.pdf,3,Partenariati Estesi PNRR.pdf,0.8157,"ai_data_digital, innovation_transfer_industry,...",
9,Piano Strategico Luiss 2021-2024 (per sito).pdf,1,PRIN 2022 PNRR.pdf,0.8000,"internationalization_collaboration, skills_edu...",SP should strengthen governance and implementa...


## Evidence Preview

Inspect full evidence, matched themes, and gaps for one strategic plan recommendation set.


In [22]:
def print_xai_for_sp(sp_file: str) -> None:
    matches = [row for row in phase2_results if row["sp_file"] == sp_file]
    if not matches:
        print(f"No Phase 2 result found for {sp_file}")
        return

    result = matches[0]
    print(f"Strategic plan: {result["sp_file"]}")

    for idx, call in enumerate(result["top_calls"], start=1):
        xai = call["xai"]
        print(f"\nRank {idx}: {xai["call_name"]} | final_score={xai["final_score"]}")
        print(f" Score breakdown: semantic={xai["score_breakdown"]["semantic_score"]}, coverage={xai["score_breakdown"]["coverage_score"]}, consistency={xai["score_breakdown"]["consistency_score"]}")

        print(" Matched themes:")
        for theme in xai["matched_themes"]:
            print(f"  - {theme["theme"]} (match={theme["match_strength"]}, call={theme["call_score"]}, sp={theme["sp_score"]})")

        print(" Key gaps:")
        if not xai["key_gaps"]:
            print("  - No critical gaps detected from current evidence.")
        else:
            for gap in xai["key_gaps"]:
                print(f"  - {gap["action"]} (gap={gap["gap_strength"]})")

        print(" Supporting chunks:")
        for ev in xai["supporting_chunks"]:
            page = ev["page"] if ev["page"] is not None else "n/a"
            print(f"  - source={ev["source_file"]}, page={page}, sim={ev["similarity"]}")
            print(f"    {ev["excerpt"][:220]}...")


print_xai_for_sp(phase2_results[0]["sp_file"])


Strategic plan: Bocconi_Piano_Strategico2021-2025&Vision2030.pdf

Rank 1: PRIN 2022 PNRR.pdf | final_score=0.7728
 Score breakdown: semantic=0.6754, coverage=1.0, consistency=1.0
 Matched themes:
  - internationalization_collaboration (match=0.2865, call=0.3333, sp=0.2865)
  - innovation_transfer_industry (match=0.059, call=0.3333, sp=0.059)
  - sustainability_climate (match=0.0243, call=0.3333, sp=0.0243)
  - ai_data_digital (match=0.0, call=0.0, sp=0.2917)
 Key gaps:
  - SP should strengthen measurable sustainability and climate actions with clear KPIs. (gap=0.309)
  - SP should strengthen technology transfer and industry engagement pathways. (gap=0.2743)
 Supporting chunks:
  - source=PRIN 2022 PNRR.pdf, page=32, sim=0.6981
    Ministero dell’Università e della Ricerca Direzione Generale della Ricerca Ufficio III 32 procedura di finanziamento, di avere beneficiato dei fondi dell’Unione Europea – Next Generation EU, componente M4C2, investimento...
  - source=PRIN 2022 PNRR.pdf, page

## Most Matched Themes

In [30]:
sp_theme_counter = Counter()
call_theme_counter = Counter()

for result in phase2_results:
    for call in result["top_calls"]:
        for theme in call["xai"].get("matched_themes", []):
            if theme.get("sp_score", 0) > 0:
                sp_theme_counter[theme["theme"]] += theme.get("sp_score", 0)

            if theme.get("call_score", 0) > 0:
                call_theme_counter[theme["theme"]] += theme.get("call_score", 0)

sp_total = sum(sp_theme_counter.values())
call_total = sum(call_theme_counter.values())

print("What universities focus on (% of total SP theme signal):")
for theme, score in sp_theme_counter.most_common():
    print(f"  {theme}: {round(score / sp_total * 100, 1)}%")

print("\nWhat funding calls focus on (% of total call theme signal):")
for theme, score in call_theme_counter.most_common():
    print(f"  {theme}: {round(score / call_total * 100, 1)}%")

What universities focus on (% of total SP theme signal):
  ai_data_digital: 31.6%
  internationalization_collaboration: 28.1%
  skills_education_capacity: 17.8%
  innovation_transfer_industry: 14.0%
  sustainability_climate: 5.0%
  governance_policy_reform: 2.4%
  inclusion_gender_social: 1.1%

What funding calls focus on (% of total call theme signal):
  innovation_transfer_industry: 31.6%
  ai_data_digital: 29.5%
  skills_education_capacity: 12.2%
  governance_policy_reform: 11.4%
  internationalization_collaboration: 7.7%
  inclusion_gender_social: 3.9%
  sustainability_climate: 3.6%


## Export for App

Export all Phase 2 results to `outputs/app_results.json` so demo strategic plans can load precomputed retrieval/XAI outputs in the Streamlit app.


In [25]:
APP_RESULTS_PATH = PROJECT_ROOT / "outputs" / "app_results.json"
APP_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

app_results = {row["sp_id"]: row for row in phase2_results}

APP_RESULTS_PATH.write_text(
    json.dumps(app_results, ensure_ascii=True, indent=2),
    encoding="utf-8",
)

print(f"Saved: {APP_RESULTS_PATH}")
print(f"Exported demo phase2 rows: {len(app_results)}")
for item in app_results.values():
    print(" -", item.get("sp_file"))


Saved: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/outputs/app_results.json
Exported demo phase2 rows: 6
 - Bocconi_Piano_Strategico2021-2025&Vision2030.pdf
 - LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf
 - Naples_Federico_Piano_strategico_2021_2026.pdf
 - Piano Strategico Luiss 2021-2024 (per sito).pdf
 - Politechnico_di_Milano_2023-2025.pdf
 - Sapienza_2021_2027.pdf


# Phase 3:  LLM Setup

Initialize local Ollama client configuration via shared llm_explainer module.


In [26]:
import importlib

import llm_explainer

importlib.reload(llm_explainer)
from llm_explainer import generate_summary

LLM_MODEL_NAME = "qwen3:0.6b"
PHASE3_TEMPERATURE = 0.1

print(f"LLM configured: {LLM_MODEL_NAME} (llm_explainer reloaded)")


LLM configured: qwen3:0.6b (llm_explainer reloaded)


## Prompt and Policy

Configure deterministic, evidence-grounded LLM summaries for executive stakeholders.


In [27]:


PHASE3_LANGUAGE = "English"
PHASE3_MIN_CITATIONS_PER_CALL = 2
PHASE3_TARGET_ACTIONS = 5
PHASE3_MAX_RETRY = 0  # stop on failure

PHASE3_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "phase3"
PHASE3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PHASE3_CONFIDENCE_RULE = (
    "Assign confidence as High/Medium/Low. If evidence is weak or citations are missing, downgrade confidence."
)

print(f"Phase 3 output dir: {PHASE3_OUTPUT_DIR}")


Phase 3 output dir: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/outputs/phase3


## LLM Generation

Generate one grounded summary per SP, stop immediately on any failure.


In [ ]:
def generate_phase3_for_sp(sp_row: Dict) -> Dict:
    llm = generate_summary(
        strategic_plan=sp_row,
        model=LLM_MODEL_NAME,
        temperature=PHASE3_TEMPERATURE,
        min_citations_per_call=PHASE3_MIN_CITATIONS_PER_CALL,
        language=PHASE3_LANGUAGE,
    )

    now_iso = datetime.now(timezone.utc).isoformat()

    result = {
        "sp_id": sp_row.get("sp_id"),
        "sp_file": sp_row.get("sp_file"),
        "model_name": LLM_MODEL_NAME,
        "timestamp": now_iso,
        "temperature": PHASE3_TEMPERATURE,
        "prompt": llm["prompt"],
        "raw_response_text": llm["raw_response_text"],
        "structured": llm["structured"],
        "stakeholder_text": llm["stakeholder_text"],
    }

    return result


phase3_results = []
for sp_row in phase2_results:
    phase3_results.append(generate_phase3_for_sp(sp_row))

print(f"Generated Phase 3 summaries for {len(phase3_results)} strategic plans")
phase3_results[0]["structured"] if phase3_results else {}



Generated Phase 3 summaries for 6 strategic plans


{'sp_id': 'Bocconi_Piano_Strategico2021-2025&Vision2030',
 'sp_file': 'Bocconi_Piano_Strategico2021-2025&Vision2030.pdf',
 'confidence': 'Medium',
 'executive_summary': "The Bocconi Strategy prioritizes digital transformation, AI-enabled education, and climate resilience. These themes are directly aligned with the SP's AI and skills priorities, while the call supports innovation transfer and international collaboration.",
 'top_calls': [{'rank': 1,
   'call_name': 'PRIN 2022 PNRR.pdf',
   'why_match': "SP emphasizes internationalization collaboration and AI-enabled education. The call explicitly funds AI/data capability building and university-industry transfer activities. Therefore this call is a medium-to-strong fit because its funded actions directly map to the SP's AI and skills priorities, though governance deliverables are less explicit in the SP.",
   'citations': [{'source_file': 'PRIN 2022 PNRR.pdf',
     'page': 32,
     'similarity': 0.6981,
     'excerpt': "Ministero dell'U

## Validation and Exports

Validate citation policy and export Phase 3 outputs for frontend/report usage.


In [ ]:
validation_rows = []

for item in phase3_results:
    structured = item.get("structured", {})
    top_calls = structured.get("top_calls", [])

    for call in top_calls:
        citations = call.get("citations", [])
        validation_rows.append(
            {
                "sp_file": item.get("sp_file"),
                "call_name": call.get("call_name"),
                "rank": call.get("rank"),
                "citation_count": len(citations),
                "citation_ok": len(citations) >= PHASE3_MIN_CITATIONS_PER_CALL,
                "confidence": call.get("confidence"),
            }
        )

phase3_validation_df = pd.DataFrame(validation_rows)

phase3_structured_path = PHASE3_OUTPUT_DIR / "phase3_structured_results.json"
phase3_text_path = PHASE3_OUTPUT_DIR / "phase3_stakeholder_texts.json"
phase3_prompts_path = PHASE3_OUTPUT_DIR / "phase3_prompts_and_raw_outputs.json"
phase3_validation_path = PHASE3_OUTPUT_DIR / "phase3_validation.csv"

phase3_structured_path.write_text(
    json.dumps([x["structured"] for x in phase3_results], ensure_ascii=True, indent=2)
)
phase3_text_path.write_text(
    json.dumps([
        {"sp_id": x["sp_id"], "sp_file": x["sp_file"], "stakeholder_text": x["stakeholder_text"]}
        for x in phase3_results
    ], ensure_ascii=True, indent=2)
)
phase3_prompts_path.write_text(
    json.dumps([
        {
            "sp_id": x["sp_id"],
            "sp_file": x["sp_file"],
            "model_name": x["model_name"],
            "timestamp": x["timestamp"],
            "prompt": x["prompt"],
            "raw_response_text": x["raw_response_text"],
        }
        for x in phase3_results
    ], ensure_ascii=True, indent=2)
)
phase3_validation_df.to_csv(phase3_validation_path, index=False)

print(f"Saved: {phase3_structured_path}")
print(f"Saved: {phase3_text_path}")
print(f"Saved: {phase3_prompts_path}")
print(f"Saved: {phase3_validation_path}")

phase3_validation_df



Saved: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/outputs/phase3/phase3_structured_results.json
Saved: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/outputs/phase3/phase3_stakeholder_texts.json
Saved: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/outputs/phase3/phase3_prompts_and_raw_outputs.json
Saved: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/outputs/phase3/phase3_validation.csv


,sp_file,call_name,rank,citation_count,citation_ok,confidence
0,Bocconi_Piano_Strategico2021-2025&Vision2030.pdf,PRIN 2022 PNRR.pdf,1,2,True,Medium
1,Bocconi_Piano_Strategico2021-2025&Vision2030.pdf,Ecosistemi dell'Innovazione PNRR.pdf,2,2,True,Medium
2,Bocconi_Piano_Strategico2021-2025&Vision2030.pdf,Partenariati Estesi PNRR.pdf,3,2,True,Medium
3,LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf,Ecosistemi dell'Innovazione PNRR.pdf,1,2,True,Medium
4,LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf,Partenariati Estesi PNRR.pdf,2,2,True,Medium
5,LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf,PRIN PNRR 2022 NextGen.pdf,3,1,False,Medium
6,Naples_Federico_Piano_strategico_2021_2026.pdf,PRIN 2022 PNRR.pdf,1,2,True,Medium
7,Naples_Federico_Piano_strategico_2021_2026.pdf,PRIN PNRR 2022 NextGen.pdf,2,3,True,Medium
8,Naples_Federico_Piano_strategico_2021_2026.pdf,Partenariati Estesi PNRR.pdf,3,3,True,Medium
9,Piano Strategico Luiss 2021-2024 (per sito).pdf,PRIN 2022 PNRR.pdf,1,2,True,Medium
